# 5. Indexing and Slicing

There are three kinds of indexing in NumPy:

1. **Basic** indexing with integers and slices. Returns a **view** of the original data.
2. **Boolean** indexing with a mask. Returns a **copy**.
3. **Fancy** indexing with lists/arrays of integers. Also returns a **copy**.

The view-vs-copy difference matters a lot once you start modifying things. There's a whole notebook on it ([12. Views and Copies](12.%20Views%20and%20Copies.ipynb)).

In [1]:
import numpy as np

arr = np.arange(100).reshape(10, 10)
arr

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

## Single elements

In [2]:
print(arr[5, 5])
print(arr[5][5])    # works too, but creates a temporary row first, so prefer arr[5, 5]
print(arr[-1, -1])  # negative indices count from the end

55
55
99


## Slicing

The syntax is `start:stop:step` in each dimension. `stop` is exclusive, and anything you leave out takes its default (start of axis, end of axis, step 1).

In [3]:
arr[1:3, 4:6]

array([[14, 15],
       [24, 25]])

In [4]:
print(arr[0])          # first row
print(arr[:, 0])       # first column
print(arr[::3, ::3])   # every third row and column
print(arr[::-1, 0])    # first column reversed

[0 1 2 3 4 5 6 7 8 9]
[ 0 10 20 30 40 50 60 70 80 90]
[[ 0  3  6  9]
 [30 33 36 39]
 [60 63 66 69]
 [90 93 96 99]]
[90 80 70 60 50 40 30 20 10  0]


Notice that `arr[:, 0]` gives a 1-D array. If you want to keep it as a column (shape `(10, 1)`), slice instead of indexing: `arr[:, 0:1]`, or `arr[:, [0]]`.

In [5]:
print(arr[:, 0].shape, arr[:, 0:1].shape)

(10,) (10, 1)


`...` (ellipsis) stands for "as many `:` as needed". It's mostly useful with higher-dimensional arrays.

In [6]:
cube = np.arange(16).reshape(2, 2, 2, 2)
print(np.array_equal(cube[0, ...], cube[0, :, :, :]))
print(cube[..., 0].shape)    # last axis fixed, all others kept

True
(2, 2, 2)


## Assigning to a slice

A scalar on the right gets broadcast to fill the whole slice:

In [7]:
grid = np.zeros((5, 6), dtype=int)
grid[1:3, :] = 1       # rows 1 and 2
grid[:, 4:] = 9        # last two columns
grid

array([[0, 0, 0, 0, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [0, 0, 0, 0, 9, 9],
       [0, 0, 0, 0, 9, 9]])

Since slices are views, assigning through them changes the original array:

In [8]:
row = grid[0]
row[:] = 7
grid

array([[7, 7, 7, 7, 7, 7],
       [1, 1, 1, 1, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [0, 0, 0, 0, 9, 9],
       [0, 0, 0, 0, 9, 9]])

## Boolean indexing

Index with a boolean array of the same length and you get the elements where it's `True`. The result is always a copy.

In [9]:
a = np.arange(25).reshape(5, 5)
keep = np.array([True, True, False, True, False])
a[keep]          # rows 0, 1 and 3

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [15, 16, 17, 18, 19]])

In [10]:
a[~keep]         # the other rows

array([[10, 11, 12, 13, 14],
       [20, 21, 22, 23, 24]])

Usually the mask comes from a condition on the data itself:

In [11]:
print(a[a > 20])                      # flattened result
print(a[(a % 2 == 0) & (a < 10)])     # even numbers below 10

[21 22 23 24]
[0 2 4 6 8]


Or from a condition on another array, for example picking the rows of a table based on one column:

In [12]:
ages = np.array([23, 35, 17, 52, 41])
salaries = np.array([30, 55, 0, 80, 62])
people = np.column_stack([ages, salaries])

people[(ages >= 18) & (salaries > 50)]

array([[35, 55],
       [52, 80],
       [41, 62]])

Boolean masks are also the simplest way to replace values:

In [13]:
b = a.copy()
b[b > 20] = 0
b

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19],
       [20,  0,  0,  0,  0]])

## Fancy indexing

Pass arrays of integers to pick arbitrary elements in any order.

In [14]:
a = np.arange(10, 20)
a[[3, 0, 3, 7]]      # repeats are fine

array([13, 10, 13, 17])

With 2-D arrays, the index arrays are **paired up**: `a[[r0, r1], [c0, c1]]` gives `a[r0, c0]` and `a[r1, c1]`, not a 2×2 block.

In [15]:
m = np.arange(100).reshape(10, 10)
m[[3, 1, 2], [3, 2, 1]]    # m[3,3], m[1,2], m[2,1]

array([33, 12, 21])

To get a **block** made of some rows and some columns, use `np.ix_`, or index in two steps:

In [16]:
rows, cols = [3, 1, 2], [6, 4, 8]
print(m[np.ix_(rows, cols)])
print(m[rows][:, cols])    # same result, but makes an extra temporary copy

[[36 34 38]
 [16 14 18]
 [26 24 28]]
[[36 34 38]
 [16 14 18]
 [26 24 28]]


## Finding elements by condition

- `np.where(cond)` returns the **indices** where `cond` is true (one array per dimension)
- `np.where(cond, x, y)` picks from `x` where true and `y` where false
- `np.argwhere(cond)` returns the indices as one `(n, ndim)` array
- `np.nonzero(a)` is the same as `np.where(a != 0)`

In [17]:
a = np.arange(16).reshape(4, 4)
print(np.where(a > 12))
print(np.argwhere(a > 12))

(array([3, 3, 3]), array([1, 2, 3]))
[[3 1]
 [3 2]
 [3 3]]


In [18]:
np.where(a > 5, -1, a)    # replace values above 5 with -1, keep the rest

array([[ 0,  1,  2,  3],
       [ 4,  5, -1, -1],
       [-1, -1, -1, -1],
       [-1, -1, -1, -1]])

In [19]:
temps = np.array([12, 25, 31, 18, 35])
np.where(temps > 30, 'hot', np.where(temps > 20, 'warm', 'cool'))

array(['cool', 'warm', 'hot', 'cool', 'hot'], dtype='<U4')

For more than two choices, `np.select` is easier to read than nested `where`:

In [20]:
conditions = [temps > 30, temps > 20]
choices = ['hot', 'warm']
np.select(conditions, choices, default='cool')

array(['cool', 'warm', 'hot', 'cool', 'hot'], dtype='<U4')

## Letting NumPy work out one dimension

Use `-1` for one dimension in `reshape` and NumPy figures it out from the total size:

In [21]:
print(np.arange(16).reshape(4, -1).shape)
print(np.arange(16).reshape(-1, 2).shape)

(4, 4)
(8, 2)
